In [2]:
!pip install xgboost

In [ ]:

# STEP 1 : IMPORT LIBRARIES


import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# XGBoost Model
from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


# STEP 2 : DOWNLOAD DATASET


path = kagglehub.dataset_download(
    "altruistdelhite04/loan-prediction-problem-dataset"
)

print("Dataset Path:", path)


# STEP 3 : LOAD DATASET


df = pd.read_csv(
    f"{path}/train_u6lujuX_CVtuZ9i.csv"
)

print(df.head())


# STEP 4 : HANDLE MISSING VALUES


df["Gender"] = df["Gender"].fillna(
    df["Gender"].mode()[0]
)

df["Married"] = df["Married"].fillna(
    df["Married"].mode()[0]
)

df["Dependents"] = df["Dependents"].fillna(
    df["Dependents"].mode()[0]
)

df["Self_Employed"] = df["Self_Employed"].fillna(
    df["Self_Employed"].mode()[0]
)

df["LoanAmount"] = df["LoanAmount"].fillna(
    df["LoanAmount"].median()
)

df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(
    df["Loan_Amount_Term"].median()
)

df["Credit_History"] = df["Credit_History"].fillna(
    df["Credit_History"].mode()[0]
)


# STEP 5 : ENCODE CATEGORICAL FEATURES


encoder = LabelEncoder()

df["Gender"] = encoder.fit_transform(
    df["Gender"]
)

df["Married"] = encoder.fit_transform(
    df["Married"]
)

df["Education"] = encoder.fit_transform(
    df["Education"]
)

df["Self_Employed"] = encoder.fit_transform(
    df["Self_Employed"]
)

df["Property_Area"] = encoder.fit_transform(
    df["Property_Area"]
)

df["Loan_Status"] = encoder.fit_transform(
    df["Loan_Status"]
)

df["Dependents"] = df["Dependents"].replace(
    "3+",
    3
)

df["Dependents"] = df["Dependents"].astype(
    int
)


# STEP 6 : REMOVE UNNECESSARY COLUMN


df.drop(
    "Loan_ID",
    axis=1,
    inplace=True
)


# STEP 7 : FEATURES AND TARGET


X = df.drop(
    "Loan_Status",
    axis=1
)

y = df["Loan_Status"]


# STEP 8 : TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# STEP 9 : CREATE XGBOOST MODEL


model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

# STEP 10 : TRAIN MODEL


model.fit(
    X_train,
    y_train
)


# STEP 11 : MAKE PREDICTIONS


y_pred = model.predict(
    X_test
)


# STEP 12 : EVALUATE MODEL


accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "\nAccuracy :",
    round(accuracy * 100, 2),
    "%"
)

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


# STEP 13 : FEATURE IMPORTANCE


importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": model.feature_importances_
    }
)

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance)


# STEP 14 : PREDICT NEW APPLICANT


new_applicant = pd.DataFrame(
    [[
        1,      # Gender
        1,      # Married
        0,      # Dependents
        1,      # Education
        0,      # Self_Employed
        5000,   # ApplicantIncome
        2000,   # CoapplicantIncome
        150,    # LoanAmount
        360,    # Loan_Amount_Term
        1,      # Credit_History
        2       # Property_Area
    ]],
    columns=X.columns
)

prediction = model.predict(
    new_applicant
)

if prediction[0] == 1:
    print("\nLoan Approved")
else:
    print("\nLoan Rejected")

100%|██████████| 12.6k/12.6k [00:00<00:00, 2.34MB/s]

Extracting files...
Dataset Path: /root/.cache/kagglehub/datasets/altruistdelhite04/loan-prediction-problem-dataset/versions/1
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area


Accuracy : 82.93 %

Confusion Matrix:
[[25 13]
 [ 8 77]]

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.66      0.70        38
           1       0.86      0.91      0.88        85

    accuracy                           0.83       123
   macro avg       0.81      0.78      0.79       123
weighted avg       0.83      0.83      0.83       123


Feature Importance:
              Feature  Importance
9      Credit_History    0.447505
8    Loan_Amount_Term    0.070404
3           Education    0.060181
6   CoapplicantIncome    0.059134
10      Property_Area    0.057757
5     ApplicantIncome    0.057756
7          LoanAmount    0.056775
2          Dependents    0.054905
1             Married    0.050100
0              Gender    0.043398
4       Self_Employed    0.042085

Loan Approved
